# Semana 10 — Armazenamento em nuvem: bloco, objeto e arquivo

Escolher o armazenamento certo é uma decisão de arquitetura. Nesta aula, comparamos os modelos de **bloco**, **objeto** e **arquivo** usando AWS como referência.

## Resultados de aprendizagem

- distinguir bloco, objeto e arquivo pelo modelo de acesso;
- relacionar EBS, S3 e EFS a problemas reais;
- escolher uma opção por latência, compartilhamento, durabilidade e custo;
- separar persistência, snapshot, backup e replicação;
- propor armazenamento para uma aplicação web.

![Fluxo de infraestrutura e aplicação em uma VPC pública](images/semana_10_fluxo_vpc_publica.png)

> O diagrama mostra o contexto de infraestrutura no qual o armazenamento persistente é usado pela aplicação.

## 1. A pergunta certa: como a aplicação acessa o dado?

| Modelo | Unidade | Acesso | Analogia |
|---|---|---|---|
| **Bloco** | blocos em um volume | dispositivo/partição | disco conectado ao servidor |
| **Objeto** | objeto + metadados + chave | API HTTP/SDK | arquivo endereçado por chave |
| **Arquivo** | arquivos e diretórios | sistema montado | pasta compartilhada |

A escolha define as operações disponíveis e o comportamento de desempenho.

## 2. Bloco: o disco da máquina

O **Amazon EBS** é anexado à EC2 e formatado com um sistema de arquivos. O sistema operacional enxerga um dispositivo e pode fazer leituras e escritas de baixa latência.

**Use para:** sistema operacional, banco de dados e workloads com escrita frequente.

**Atenção:** persistência não é backup; alta disponibilidade exige snapshots, replicação no banco ou serviço gerenciado.

In [ ]:
# Exemplo conceitual em Linux (não execute neste notebook):
comandos = ['lsblk', 'sudo mkfs -t ext4 /dev/xvdf', 'sudo mount /dev/xvdf /dados', 'df -h /dados']
print('\n'.join(comandos))

## 3. Objeto: dados acessados por API

No **Amazon S3**, cada objeto combina conteúdo, metadados e uma chave em um bucket. A aplicação usa API, SDK ou CLI, e não um disco tradicional.

**Use para:** imagens, vídeos, documentos, backups, datasets e conteúdo estático.

Pontos fortes: durabilidade, escala, versionamento, políticas de ciclo de vida e integração com eventos.

In [ ]:
objetos = [
    {'chave': 'clientes/42/avatar.png', 'classe': 'STANDARD'},
    {'chave': 'backups/2026-10-08.sql.gz', 'classe': 'GLACIER'},
]
for objeto in objetos:
    print(objeto['chave'], '->', objeto['classe'])

## 4. Arquivo: diretórios compartilhados

O **Amazon EFS** fornece um sistema de arquivos gerenciado que pode ser montado simultaneamente por várias instâncias Linux.

**Use para:** uploads compartilhados, arquivos de configuração ou aplicações que exigem caminhos conhecidos.

Compartilhamento não transforma o EFS em banco de dados nem elimina controle de concorrência, permissões e retenção.

## 5. Comparação para decidir

| Critério | Bloco (EBS) | Objeto (S3) | Arquivo (EFS) |
|---|---|---|---|
| Acesso | dispositivo | API/HTTP/SDK | montagem NFS |
| Compartilhamento | geralmente uma instância | via API | várias instâncias |
| Melhor uso | SO e banco | conteúdo e backup | diretório comum |
| Pergunta-chave | preciso de disco? | acesso por chave basta? | preciso montar diretórios? |

Regra prática: **banco em bloco; arquivos e backups em objeto; diretório compartilhado em arquivo** — valide requisitos antes de decidir.

## 6. Persistência, backup e durabilidade

- **Persistência:** o dado sobrevive ao reinício do processo ou da instância.
- **Snapshot/backup:** cópia usada para recuperação; não substitui produção.
- **Replicação:** cópias para disponibilidade e recuperação.
- **Versionamento:** preserva versões de objetos e ajuda contra exclusões acidentais.

Um EBS persistente pode sobreviver à terminação da EC2 se configurado corretamente, mas isso não significa que haja backup. No S3, versionamento e ciclo de vida ajudam a proteger dados e controlar custos.

## 7. Estudo de caso: aplicação web de produtos

1. banco relacional em serviço gerenciado ou volume de bloco;
2. imagens no S3, guardando no banco apenas a chave do objeto;
3. EFS apenas se várias instâncias precisarem de diretório compartilhado;
4. backups automatizados, retenção e IAM de menor privilégio.

Salvar imagens somente no disco da EC2 é frágil: a instância pode ser substituída, escalada ou falhar.

## Atividade de projeto

Escolha bloco, objeto ou arquivo para: (1) PostgreSQL transacional; (2) catálogo de imagens; (3) uploads compartilhados por duas instâncias; (4) backup mensal raramente recuperado.

Justifique com modelo de acesso, latência, compartilhamento, backup e risco de indisponibilidade.

## Síntese

- **Bloco**: semântica de disco para SO e bancos.
- **Objeto**: API, metadados, escala e durabilidade para conteúdo e backups.
- **Arquivo**: diretórios compartilhados.
- A decisão combina acesso, desempenho, disponibilidade, segurança e custo.

### Referências
AWS EBS: https://docs.aws.amazon.com/ebs/
AWS S3: https://docs.aws.amazon.com/s3/
AWS EFS: https://docs.aws.amazon.com/efs/